# SCC.454 Programming Quiz: Recommender Systems with Apache Spark

**Duration:** 30 minutes  
**Module:** SCC.454 – Large Scale Platforms for AI and Data Analysis  
**Lecturer:** Dr Tharindu Ranasinghe  
**Lancaster University – School of Computing and Communications**

---

## Instructions

- This quiz contains **one question** with **four sub-questions** (a–d), totalling **100 marks**.
- Fill in the **empty code cells** marked with `# YOUR CODE HERE`.
- You may refer to the **PySpark API documentation** (links provided below).
- All questions are **coding only** — no written explanations required.
- The setup cells (Spark installation, data loading, and helper code) are provided — **run them first** before attempting the questions.
- **Do not modify** the provided setup cells.

---

## Mark Allocation

| Sub-question | Topic | Marks |
|:---:|:---:|:---:|
| **Q1(a)** | Data Exploration with Spark | **20** |
| **Q1(b)** | Content-Based Filtering | **25** |
| **Q1(c)** | Collaborative Filtering (ALS) | **30** |
| **Q1(d)** | Evaluation Metrics | **25** |
| | **Total** | **100** |

---

# Setting Up Apache Spark

Before starting the quiz, you need to install Apache Spark and its dependencies. Run the cells below **once** at the start.

### Installation Steps

**Step 1:** Install Java (Spark requires Java 11+) and PySpark via pip.

**Step 2:** Set the `JAVA_HOME` environment variable so Spark can locate Java.

**Step 3:** Create a `SparkSession` — this is the entry point for all Spark operations.

> **Note:** If you are running this on **Google Colab**, the cells below will work as-is.  
> If you are running **locally**, make sure Java 11+ is installed and `JAVA_HOME` is set in your system environment.

### PySpark API Documentation

You may consult the following documentation during this quiz:

- **PySpark SQL & DataFrames:** [https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.sql/index.html](https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.sql/index.html)
- **PySpark ML (Machine Learning):** [https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.ml.html](https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.ml.html)
- **ALS (Alternating Least Squares):** [https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.recommendation.ALS.html](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.recommendation.ALS.html)
- **pyspark.sql.functions:** [https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.sql/functions.html](https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.sql/functions.html)
- **RegressionEvaluator:** [https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)
- **TF-IDF (HashingTF + IDF):** [https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.feature.HashingTF.html](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.feature.HashingTF.html)

Install Spark according to your platform

In [1]:
# # Cell 1: Install dependencies (run once)
# !pip install pandas matplotlib seaborn numpy -q
# !pip install pyspark==3.5.0 -q
# !apt-get update -qq > /dev/null 2>&1
# !apt-get install openjdk-11-jdk-headless -qq > /dev/null 2>&1

# import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# print("PySpark and Java installed successfully!")

In [2]:
import os, sys
from pyspark.sql import SparkSession

# Force Spark to use *this* notebook's Python (your venv)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

---

# Setup — Run All Cells Below Before Starting the Quiz

⚠️ **Do NOT modify these cells.** Just run them in order.

In [3]:
# Cell 2: Imports and Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import HashingTF, IDF, Tokenizer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.linalg import Vectors
import pandas as pd
import numpy as np
import os, urllib.request, zipfile

spark = SparkSession.builder \
    .appName("SCC454-Quiz") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("Spark session ready!")

Spark version: 3.5.0
Spark session ready!


In [4]:
# Cell 3: Download MovieLens dataset
data_dir = "data"
ml_dir = os.path.join(data_dir, "ml-latest-small")

if not os.path.exists(ml_dir):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zip_path = os.path.join(data_dir, "ml-latest-small.zip")
    print("Downloading MovieLens dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    os.remove(zip_path)
    print("Done!")
else:
    print(f"Dataset already exists at {ml_dir}")

Dataset already exists at data\ml-latest-small


In [5]:
# Cell 4: Load data into Spark DataFrames
ratings = spark.read.csv(os.path.join(ml_dir, "ratings.csv"), header=True, inferSchema=True)
movies = spark.read.csv(os.path.join(ml_dir, "movies.csv"), header=True, inferSchema=True)
tags = spark.read.csv(os.path.join(ml_dir, "tags.csv"), header=True, inferSchema=True)

print(f"Ratings: {ratings.count():,} rows")
print(f"Movies:  {movies.count():,} rows")
print(f"Tags:    {tags.count():,} rows")
print("\nRatings schema:")
ratings.printSchema()
print("Movies schema:")
movies.printSchema()

Ratings: 100,836 rows
Movies:  9,742 rows
Tags:    3,683 rows

Ratings schema:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

Movies schema:
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [6]:
# Cell 5: Build TF-IDF item profiles (provided for use in Q1b)
# Combine genres and tags into content text per movie
movie_tags = tags.groupBy("movieId").agg(
    F.concat_ws(" ", F.collect_list(F.lower("tag"))).alias("all_tags")
)
movie_content = movies.withColumn(
    "genre_text", F.lower(F.regexp_replace("genres", "\\|", " "))
)
movie_content = movie_content.join(movie_tags, "movieId", "left").withColumn(
    "content_text",
    F.concat_ws(" ", F.col("genre_text"), F.coalesce(F.col("all_tags"), F.lit("")))
)

# TF-IDF pipeline
tokenizer = Tokenizer(inputCol="content_text", outputCol="words")
movie_words = tokenizer.transform(movie_content)
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=256)
movie_tf = hashingTF.transform(movie_words)
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
idf_model = idf.fit(movie_tf)
movie_tfidf = idf_model.transform(movie_tf)
movie_tfidf = movie_tfidf.select("movieId", "title", "genres", "tfidf_features").cache()

print("TF-IDF item profiles computed!")
print(f"Number of movies with TF-IDF profiles: {movie_tfidf.count():,}")

TF-IDF item profiles computed!
Number of movies with TF-IDF profiles: 9,742


In [7]:
# Cell 6: Prepare data for ALS (provided for use in Q1c and Q1d)
ratings_als = ratings.select(
    F.col("userId").cast("integer"),
    F.col("movieId").cast("integer"),
    F.col("rating").cast("float")
)

(training, test) = ratings_als.randomSplit([0.8, 0.2], seed=42)
training.cache()
test.cache()

print(f"Training ratings: {training.count():,}")
print(f"Test ratings:     {test.count():,}")

Training ratings: 80,578
Test ratings:     20,258


---

# ✅ Setup complete — Begin the quiz below

---

# Question 1: Recommender Systems with Apache Spark [100 marks]

---

## Q1(a): Data Exploration with Spark [20 marks]

In the MovieLens dataset, each movie has a `genres` field containing one or more genres separated by `|` (e.g., `"Action|Adventure|Sci-Fi"`).

**Task:** Write PySpark code to produce a table showing, for **each individual genre**, the following statistics:

1. `movie_count` — How many movies belong to that genre  
2. `avg_rating` — The average rating of all ratings for movies in that genre (rounded to 2 decimal places)  
3. `total_ratings` — The total number of ratings for movies in that genre  

Show only the **top 8 genres** sorted by `movie_count` descending.

**Expected output:**

| genre | movie_count | avg_rating | total_ratings |
|:---:|:---:|:---:|:---:|
| Drama | 1053 | 3.68 | 30,655 |
| Comedy | 795 | 3.44 | 18,187 |
| Thriller | 416 | 3.52 | 13,812 |
| Romance | 380 | 3.51 | 9,949 |
| Action | 374 | 3.47 | 14,625 |
| Horror | 250 | 3.29 | 5,181 |
| Crime | 246 | 3.69 | 9,376 |
| Documentary | 213 | 3.75 | 2,587 |

**Hints:**
- Use `F.explode()` to split the genre array into individual rows — see [pyspark.sql.functions.explode](https://spark.apache.org/docs/3.5.0/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)
- Use `F.split()` to convert the pipe-separated genres string into an array
- You will need to join the `movies` and `ratings` DataFrames

In [8]:
from pyspark.sql import functions as func

# Split genres into individual rows
movie_genres = movies.select(
    "movieId",
    func.explode(func.split(func.col("genres"), "\\|")).alias("genre")
)

# Join with ratings
genre_ratings = movie_genres.join(ratings, "movieId")

# Compute statistics
genre_stats = genre_ratings.groupBy("genre").agg(
    func.countDistinct("movieId").alias("movie_count"),
    func.round(func.avg("rating"), 2).alias("avg_rating"),
    func.count("rating").alias("total_ratings")
)

# Order for readability
genre_stats = genre_stats.orderBy(func.desc("movie_count"))

genre_stats.show(8, truncate=False)


+---------+-----------+----------+-------------+
|genre    |movie_count|avg_rating|total_ratings|
+---------+-----------+----------+-------------+
|Drama    |4349       |3.66      |41928        |
|Comedy   |3753       |3.38      |39053        |
|Thriller |1889       |3.49      |26452        |
|Action   |1828       |3.45      |30635        |
|Romance  |1591       |3.51      |18124        |
|Adventure|1262       |3.51      |24161        |
|Crime    |1196       |3.66      |16681        |
|Sci-Fi   |980        |3.46      |17243        |
+---------+-----------+----------+-------------+
only showing top 8 rows



---

## Q1(b): Content-Based Filtering — Cosine Similarity [25 marks]

The TF-IDF item profiles have been pre-computed for you in the setup (stored in the `movie_tfidf` DataFrame).

**Task (Part i — 20 marks):** Complete the function below by filling in the **four blanks** (`___BLANK_1___` through `___BLANK_4___`). Each blank is worth **5 marks**. The function should find the top-N most similar movies to a given target movie using **cosine similarity** on TF-IDF vectors.

```python
def find_top_similar(target_title, movie_tfidf_df, top_n=5):
    """Find top-N similar movies using cosine similarity."""

    # Step 1: Get the target movie row
    target = movie_tfidf_df.filter(
        F.col("title").contains(target_title)
    ).first()

    target_vec = target["tfidf_features"]
    target_id  = target["movieId"]

    # Step 2: Collect all OTHER movies (exclude the target)
    others = movie_tfidf_df.filter(
        ___BLANK_1___                    # [5 marks]
    ).collect()

    # Step 3: Compute cosine similarity for each candidate
    sims = []
    for row in others:
        a = target_vec.toArray()
        b = ___BLANK_2___               # [5 marks]
        dot   = float(np.dot(a, b))
        norms = float(np.linalg.norm(a) * np.linalg.norm(b))
        sim   = ___BLANK_3___           # [5 marks]
        sims.append((row["title"], row["genres"], sim))

    # Step 4: Sort descending by similarity & return top_n
    sims.sort(key=lambda x: ___BLANK_4___, reverse=True)  # [5 marks]
    return sims[:top_n]
```

In [9]:
def find_top_similar(target_title, movie_tfidf_df, top_n=5):
    """Find top-N similar movies using cosine similarity."""

    # Step 1: Get the target movie row
    target = movie_tfidf_df.filter(
        F.col("title").contains(target_title)
    ).first()

    target_vec = target["tfidf_features"]
    target_id  = target["movieId"]

    # Step 2: Collect all OTHER movies (exclude the target)
    others = movie_tfidf_df.filter(
        F.col("movieId") != target_id     # BLANK 1
    ).collect()

    # Step 3: Compute cosine similarity for each candidate
    sims = []
    for row in others:
        a = target_vec.toArray()
        b = row["tfidf_features"].toArray() # BLANK 2
        dot   = float(np.dot(a, b))
        norms = float(np.linalg.norm(a) * np.linalg.norm(b))
        sim   = dot / norms if norms != 0 else 0.0 # BLANK 3
        sims.append((row["title"], row["genres"], sim))

    # Step 4: Sort descending by similarity & return top_n
    sims.sort(key=lambda x: x[2], reverse=True)  # BLANK 4 
    return sims[:top_n]

**Task (Part ii — 5 marks):** Call your completed function to find the **top 5 movies most similar** to `"Forrest Gump"`. Print the results showing title, genres, and similarity score.

In [10]:
similar_movies = find_top_similar("Forrest Gump", movie_tfidf)

for i, (title, genres, sim) in enumerate(similar_movies, 1):
    print(f"  {i:2d}. {title:50s} [{genres:30s}] sim={sim:.4f}")

   1. Waterboy, The (1998)                               [Comedy                        ] sim=0.4567
   2. Necessary Roughness (1991)                         [Comedy                        ] sim=0.4567
   3. Rudy (1993)                                        [Drama                         ] sim=0.4567
   4. Great Santini, The (1979)                          [Drama                         ] sim=0.4567
   5. Return, The (Vozvrashcheniye) (2003)               [Drama                         ] sim=0.4567


---

## Q1(c): Collaborative Filtering with ALS [30 marks]

The training and test sets have been pre-split for you in the setup cells (`training` and `test` DataFrames).

**Task (Part i — 10 marks):** Train an ALS model using **exactly** the parameters shown in the configuration table below.

| Parameter | Value | Notes |
|:---:|:---:|:---|
| `rank` | `30` | Number of latent factors |
| `maxIter` | `15` | Training iterations |
| `regParam` | `0.05` | L2 regularisation |
| `userCol` | `"userId"` | Column name for user IDs |
| `itemCol` | `"movieId"` | Column name for item IDs |
| `ratingCol` | `"rating"` | Column name for ratings |
| `coldStartStrategy` | `"drop"` | Handle missing predictions |
| `seed` | `123` | Random seed for reproducibility |

- **API Reference:** [pyspark.ml.recommendation.ALS](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.recommendation.ALS.html)

In [11]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=30,
    maxIter=15,
    regParam=0.05,
    coldStartStrategy="drop",
    seed=123
)

als_model = als.fit(training)


**Task (Part ii — 10 marks):** Generate the **top 10 movie recommendations** for **User 75**. Display the results as a DataFrame showing the `title`, `genres`, and `predicted_score` columns, ordered by predicted score descending.

For reference, here is a sample of User 75's actual ratings from the dataset:

| movieId | title | genres | rating |
|:---:|:---|:---|:---:|
| 1 | Toy Story (1995) | Adventure\|Animation\|... | 4.0 |
| 32 | Twelve Monkeys (1995) | Mystery\|Sci-Fi\|Thriller | 4.0 |
| 47 | Seven (a.k.a. Se7en) (1995) | Mystery\|Thriller | 5.0 |
| 50 | Usual Suspects, The (1995) | Crime\|Mystery\|Thriller | 5.0 |
| 110 | Braveheart (1995) | Action\|Drama\|War | 5.0 |
| 150 | Apollo 13 (1995) | Adventure\|Drama\|IMAX | 3.0 |
| 260 | Star Wars: Ep. IV (1977) | Action\|Adventure\|Drama | 5.0 |
| 296 | Pulp Fiction (1994) | Comedy\|Crime\|Drama\|... | 5.0 |

In [12]:
from pyspark.sql import functions as F

# Create DataFrame containing user 75
user_df = spark.createDataFrame([(75,)], ["userId"])

# Generate top 10 recommendations
recs = als_model.recommendForUserSubset(user_df, 10)

# Explode the recommendation list
recs_exploded = recs.select(
    F.col("userId"),
    F.explode("recommendations").alias("rec")
).select(
    F.col("rec.movieId"),
    F.col("rec.rating").alias("predicted_score")
)

# Join with movies to get titles and genres
top_movies = recs_exploded.join(
    movies,
    "movieId"
).select(
    "title",
    "genres",
    "predicted_score"
).orderBy(
    F.desc("predicted_score")
)

# Display results
top_movies.show(10, truncate=False)

+----------------------------------------------------+--------------------------------------------------+---------------+
|title                                               |genres                                            |predicted_score|
+----------------------------------------------------+--------------------------------------------------+---------------+
|Wallace & Gromit: The Wrong Trousers (1993)         |Animation|Children|Comedy|Crime                   |5.688599       |
|Aristocrats, The (2005)                             |Comedy|Documentary                                |5.3239365      |
|Wallace & Gromit: A Close Shave (1995)              |Animation|Children|Comedy                         |5.137681       |
|Third Man, The (1949)                               |Film-Noir|Mystery|Thriller                        |4.886505       |
|Patton (1970)                                       |Drama|War                                         |4.7104464      |
|The Lego Movie (2014)  

**Task (Part iii — 10 marks):** Using the ALS model's **item factors** (`als_model.itemFactors`), find the **top 5 movies most similar** to `"Star Wars: Episode IV - A New Hope (1977)"` in the **latent factor space** using cosine similarity.

Display the title, genres, and cosine similarity score for each result.

- **Hint:** The item factors DataFrame has columns `id` (movieId) and `features` (latent vector). You will need to compute cosine similarity between the target movie's latent vector and all other movies' latent vectors.

In [13]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import numpy as np

# Get movieId for the target movie
target_movie = movies.filter(
    F.col("title") == "Star Wars: Episode IV - A New Hope (1977)"
).select("movieId").first()

target_id = target_movie["movieId"]

# Get the latent vector of the target movie
target_vec = als_model.itemFactors.filter(
    F.col("id") == target_id
).select("features").first()["features"]

target_array = np.array(target_vec)

# Cosine similarity function
def cosine_sim(vec):
    b = np.array(vec)
    dot = float(np.dot(target_array, b))
    norms = float(np.linalg.norm(target_array) * np.linalg.norm(b))
    return dot / norms if norms != 0 else 0.0

cosine_udf = F.udf(cosine_sim, DoubleType())

# Compute similarity for all movies
similar_movies = als_model.itemFactors.withColumn(
    "similarity",
    cosine_udf(F.col("features"))
)

# Step 5: Join with movie metadata and exclude target movie
results = similar_movies.join(
    movies,
    similar_movies.id == movies.movieId
).filter(
    F.col("movieId") != target_id
).select(
    "title",
    "genres",
    "similarity"
).orderBy(
    F.desc("similarity")
).limit(5)

# Display results
results.show(truncate=False)

+-----------------------------------------------------+-----------------------+------------------+
|title                                                |genres                 |similarity        |
+-----------------------------------------------------+-----------------------+------------------+
|Star Wars: Episode V - The Empire Strikes Back (1980)|Action|Adventure|Sci-Fi|0.9591302684247586|
|Stalag 17 (1953)                                     |Drama|War              |0.9131579744915458|
|Star Wars: Episode VI - Return of the Jedi (1983)    |Action|Adventure|Sci-Fi|0.9029042725927396|
|Enter the Dragon (1973)                              |Action|Crime           |0.89510817606596  |
|Soldier (1998)                                       |Action|Sci-Fi|War      |0.8945946557543096|
+-----------------------------------------------------+-----------------------+------------------+



**Task (Part iv - (25 marks):** Compute the **Test RMSE** of your ALS model on the test set.

- Use `als_model.transform(test)` to generate predictions.
- Use `RegressionEvaluator` with `metricName="rmse"` — see [API docs](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html).
- Print the result as: `Test RMSE: <value>`

In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions
predictions = als_model.transform(test)

# Create RMSE evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Compute RMSE
rmse = evaluator.evaluate(predictions)

# Print result
print(f"Test RMSE: {rmse}")

Test RMSE: 0.949205046052884


---

# EXTRA PART IN ANSWERS

## Q1(d): Evaluation Metrics [25 marks]

Using the ALS model you trained in Q1(c), compute the following evaluation metrics on the `test` set.

Your output should present the following four metrics:

| Metric | Value |
|:---|:---:|
| Test RMSE | *< your result >* |
| Precision@5 | *< your result >* |
| Precision@10 | *< your result >* |
| Catalog Coverage (K=10) | *< your result >* |

**Task (Part i — 10 marks):** Compute the **Test RMSE** of your ALS model on the test set.

- Use `als_model.transform(test)` to generate predictions.
- Use `RegressionEvaluator` with `metricName="rmse"` — see [API docs](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html).
- Print the result as: `Test RMSE: <value>`

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions
predictions = als_model.transform(test)

# Create RMSE evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Compute RMSE
rmse = evaluator.evaluate(predictions)

# Print result
print(f"Test RMSE: {rmse}")

Test RMSE: 0.949205046052884


**Task (Part ii — 10 marks):** Compute **Precision@5** and **Precision@10** for the ALS model.

A recommendation is considered **relevant** if the user rated that movie **≥ 4.0** in the test set.

$$\text{Precision@K} = \frac{\text{Number of relevant items in top-K recommendations}}{K}$$

Report the **average Precision@K across all users** who have at least one relevant item in the test set.

- **Hint:** Use `als_model.recommendForAllUsers(K)` to get top-K recommendations, then join with relevant test items.
- Print results as: `Precision@5: <value>` and `Precision@10: <value>`

In [16]:
from pyspark.sql import functions as F

def precision_at_k(model, test_df, k):
    
    # Relevant items (rating >= 4.0)
    relevant = test_df.filter(F.col("rating") >= 4.0) \
        .select("userId", "movieId")

    # Top-K recommendations
    recs = model.recommendForAllUsers(k)

    # Flatten recommendations
    recs_flat = recs.select(
        "userId",
        F.explode("recommendations").alias("rec")
    ).select(
        "userId",
        F.col("rec.movieId").alias("movieId")
    )

    # Join with relevant items
    hits = recs_flat.join(relevant, ["userId", "movieId"])

    # Count hits per user
    hits_per_user = hits.groupBy("userId").count() \
        .withColumnRenamed("count", "hits")

    # Precision per user
    precision = hits_per_user.withColumn(
        "precision",
        F.col("hits") / k
    )

    # Average precision
    return precision.select(F.avg("precision")).first()[0]


p5 = precision_at_k(als_model, test, 5)
p10 = precision_at_k(als_model, test, 10)

print(f"Precision@5: {p5:.4f}")
print(f"Precision@10: {p10:.4f}")

Precision@5: 0.2270
Precision@10: 0.1178


**Task (Part iii — 5 marks):** Compute the **Catalog Coverage** at K=10.

$$\text{Coverage} = \frac{\text{Number of unique movies recommended across all users}}{\text{Total number of movies in catalog}}$$

- Use `als_model.recommendForAllUsers(10)` and count the distinct recommended movieIds.
- Print the result as: `Catalog Coverage (K=10): <value>%`

In [17]:
from pyspark.sql import functions as F

# Get top-10 recommendations for all users
recs = als_model.recommendForAllUsers(10)

# Extract movieIds
recommended_movies = recs.select(
    F.explode("recommendations").alias("rec")
).select(
    F.col("rec.movieId").alias("movieId")
).distinct()

num_recommended = recommended_movies.count()
total_movies = movies.select("movieId").distinct().count()

coverage = (num_recommended / total_movies) * 100

print(f"Catalog Coverage (K=10): {coverage:.2f}%")

Catalog Coverage (K=10): 14.64%


---

# End of Quiz

**Remember to save your notebook before submitting.**

When you are finished, stop the Spark session by running the cell below.

In [18]:
# Stop Spark session
spark.stop()
print("Spark session stopped. Quiz complete!")

Spark session stopped. Quiz complete!
